# 05 · Beachwatch Site Registry (Phase 1)

Fetch Beachwatch sites, extract geographic features via GEE, QA, and export.

> **Prerequisite:** `00_setup.ipynb`

In [ ]:
import sys
if '/content/geo_sp_project/src' not in sys.path:
    sys.path.extend(['/content/geo_sp_project/src', '/content/geo_sp_project/configs'])
    import config; from geo_sp.auth import init_ee; init_ee(config.EE_PROJECT)


In [ ]:
import geemap
import config
from geo_sp.beachwatch import (
    fetch_sites, build_geodataframe, build_buffers,
    save_registry, save_buffer_csv, load_gee_layers,
    export_registry_to_drive, qa_check, build_site_map
)

df_sites, gj = fetch_sites(config.BEACHWATCH_GEOJSON_URL)
print(f'\nFetched {len(df_sites)} sites')
print(df_sites[['site_id','site_name','lat','lon']].head(3))


In [ ]:
# QA + save basic registry
qa = qa_check(df_sites)
for k, v in qa.items(): print(f'  {k}: {v}')

save_registry(df_sites, config.SITE_REGISTRY_CSV, config.SITE_REGISTRY_PARQUET)

gdf = build_geodataframe(df_sites)
gdf_buf = build_buffers(gdf)
save_buffer_csv(gdf_buf, config.SITE_BUFFER_CSV)


In [ ]:
# Load GEE layers + start export task
import geemap
fc = geemap.geojson_to_ee(gj)
layers = load_gee_layers()

# Uncomment to export to Google Drive (takes 5-15 min):
# export_registry_to_drive(fc, layers)


In [ ]:
# Interactive map
m = build_site_map(fc, layers['dem'], [-33.8, 151.2], 9)
m
